In [ ]:
# Parameters - These will be injected by the pipeline
workspace_id = ""
dataset_id = ""
semantic_model_name = ""
polling_interval = "15"
max_wait_minutes = "30"

In [ ]:
# Cell 1: Import Libraries and Setup
from notebookutils import mssparkutils
import requests
import json
import sys
import os
import time
from datetime import datetime, timedelta

print("=" * 80)
print("🔄 UNIVERSAL SEMANTIC MODEL REFRESH - WITH POLLING")
print("=" * 80)
print(f"Notebook started at: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print()


In [ ]:
# Cell 2: Validate Parameters

print("=" * 80)
print("📋 CHECKING PARAMETERS")
print("=" * 80)

# Convert string parameters to proper types
polling_interval = int(polling_interval) if polling_interval else 15
max_wait_minutes = int(max_wait_minutes) if max_wait_minutes else 30

print(f"📊 Semantic Model: {semantic_model_name if semantic_model_name else '❌ MISSING'}")
print(f"📁 Workspace ID:   {workspace_id if workspace_id else '❌ MISSING'}")
print(f"🔢 Dataset ID:     {dataset_id if dataset_id else '❌ MISSING'}")
print(f"⏱️  Poll Interval:  {polling_interval} seconds")
print(f"⏰ Max Wait Time:  {max_wait_minutes} minutes")
print()

# Validate required parameters
if not workspace_id or not dataset_id:
    error_msg = "❌ ERROR: Missing required parameters (workspace_id and/or dataset_id)"
    print(error_msg)
    print("\nReceived parameters:")
    print(f"  workspace_id = '{workspace_id}'")
    print(f"  dataset_id = '{dataset_id}'")
    print(f"  semantic_model_name = '{semantic_model_name}'")
    raise ValueError(error_msg)

print("✅ All required parameters present")
print()

In [ ]:
# Cell 3: Authenticate to Power BI

print("=" * 80)
print("🔐 AUTHENTICATION")
print("=" * 80)

try:
    token = mssparkutils.credentials.getToken("pbi")
    
    headers = {
        "Authorization": f"Bearer {token}",
        "Content-Type": "application/json"
    }
    
    print("✅ Power BI authentication successful")
    print()
    
except Exception as e:
    error_msg = f"❌ Authentication failed: {str(e)}"
    print(error_msg)
    raise Exception(error_msg)

In [ ]:
# Cell 4: Trigger Semantic Model Refresh

print("=" * 80)
print("🚀 TRIGGERING SEMANTIC MODEL REFRESH")
print("=" * 80)

refresh_url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{dataset_id}/refreshes"

print(f"📊 Model: {semantic_model_name}")
print(f"🔗 Endpoint: {refresh_url}")
print()
print("⏳ Initiating refresh request...")

refresh_start_time = datetime.now()

try:
    # Trigger the refresh
    response = requests.post(
        refresh_url, 
        headers=headers, 
        json={"notifyOption": "NoNotification"}
    )
    
    if response.status_code in [200, 202]:
        print(f"✅ Refresh request accepted (Status: {response.status_code})")
        print(f"⏰ Request sent at: {refresh_start_time.strftime('%H:%M:%S')}")
        print()
        
        # Extract request ID from response headers if available
        request_id = response.headers.get('RequestId', 'Unknown')
        print(f"🆔 Request ID: {request_id}")
        print()
        
    else:
        error_msg = f"❌ FAILED: Status {response.status_code} - {response.text}"
        print(error_msg)
        raise Exception(error_msg)
        
except Exception as e:
    error_msg = f"❌ ERROR triggering refresh: {str(e)}"
    print(error_msg)
    raise Exception(error_msg)

In [ ]:
# Cell 5: Poll Refresh Status Until Complete

print("=" * 80)
print("🔍 POLLING REFRESH STATUS")
print("=" * 80)
print(f"⏱️  Checking status every {polling_interval} seconds")
print(f"⏰ Maximum wait time: {max_wait_minutes} minutes")
print(f"🎯 Looking for refresh started at: {refresh_start_time.strftime('%H:%M:%S')}")
print()

# Calculate timeout
max_wait_time = timedelta(minutes=max_wait_minutes)
timeout_time = refresh_start_time + max_wait_time

# Status tracking
refresh_completed = False
refresh_status = "Unknown"
refresh_error = None
poll_count = 0
last_status = None

# Wait a few seconds before first poll (give API time to register the refresh)
print("⏳ Waiting 5 seconds before first status check...")
time.sleep(5)
print()

try:
    while not refresh_completed:
        poll_count += 1
        current_time = datetime.now()
        elapsed_time = current_time - refresh_start_time
        elapsed_minutes = int(elapsed_time.total_seconds() / 60)
        elapsed_seconds = int(elapsed_time.total_seconds() % 60)
        
        # Check for timeout
        if current_time > timeout_time:
            error_msg = f"❌ TIMEOUT: Refresh did not complete within {max_wait_minutes} minutes"
            print()
            print(error_msg)
            raise Exception(error_msg)
        
        # Get refresh history
        history_url = f"https://api.powerbi.com/v1.0/myorg/groups/{workspace_id}/datasets/{dataset_id}/refreshes?$top=5"
        
        try:
            history_response = requests.get(history_url, headers=headers)
            
            if history_response.status_code != 200:
                print(f"⚠️  Poll #{poll_count}: Could not retrieve refresh history (Status: {history_response.status_code})")
                time.sleep(polling_interval)
                continue
            
            history_data = history_response.json()
            refreshes = history_data.get('value', [])
            
            if not refreshes:
                print(f"⏳ Poll #{poll_count}: No refresh history found yet... (Elapsed: {elapsed_minutes}m {elapsed_seconds}s)")
                time.sleep(polling_interval)
                continue
            
            # Get the most recent refresh (first in the list)
            latest_refresh = refreshes[0]
            refresh_status = latest_refresh.get('status', 'Unknown')
            refresh_type = latest_refresh.get('refreshType', 'Unknown')
            start_time_str = latest_refresh.get('startTime', '')
            end_time_str = latest_refresh.get('endTime', '')
            
            # Parse the start time to make sure this is OUR refresh
            if start_time_str:
                try:
                    # Power BI returns ISO format: 2025-11-26T15:30:45.123Z
                    api_start_time = datetime.strptime(start_time_str.split('.')[0], '%Y-%m-%dT%H:%M:%S')
                    time_diff = abs((api_start_time - refresh_start_time).total_seconds())
                    
                    # If the refresh started more than 2 minutes before our trigger, it's not ours
                    if time_diff > 120:
                        print(f"⏳ Poll #{poll_count}: Waiting for our refresh to appear... (Elapsed: {elapsed_minutes}m {elapsed_seconds}s)")
                        time.sleep(polling_interval)
                        continue
                except:
                    pass  # If we can't parse, assume it's ours
            
            # Check refresh status
            if refresh_status != last_status:
                # Status changed, print update
                status_icon = {
                    'Unknown': '❓',
                    'InProgress': '⏳',
                    'Completed': '✅',
                    'Failed': '❌',
                    'Cancelled': '🚫'
                }.get(refresh_status, '❓')
                
                print(f"{status_icon} Poll #{poll_count}: Status = {refresh_status} | Type = {refresh_type} | Elapsed: {elapsed_minutes}m {elapsed_seconds}s")
                last_status = refresh_status
            else:
                # Same status, just print a dot to show we're still checking
                print(f"   Poll #{poll_count}: {refresh_status}... ({elapsed_minutes}m {elapsed_seconds}s)", end='\r')
            
            # Check if refresh is complete
            if refresh_status == 'Completed':
                refresh_completed = True
                print()  # New line after the \r
                print()
                print("─" * 80)
                print("✅ REFRESH COMPLETED SUCCESSFULLY")
                print("─" * 80)
                
                if end_time_str:
                    end_time = datetime.strptime(end_time_str.split('.')[0], '%Y-%m-%dT%H:%M:%S')
                    actual_duration = end_time - api_start_time if 'api_start_time' in locals() else elapsed_time
                    duration_seconds = int(actual_duration.total_seconds())
                    duration_minutes = duration_seconds // 60
                    duration_seconds_remainder = duration_seconds % 60
                    
                    print(f"📊 Model: {semantic_model_name}")
                    print(f"⏱️  Duration: {duration_minutes}m {duration_seconds_remainder}s")
                    print(f"🏁 Completed at: {end_time_str}")
                else:
                    print(f"📊 Model: {semantic_model_name}")
                    print(f"⏱️  Total elapsed time: {elapsed_minutes}m {elapsed_seconds}s")
                
                print()
                break
                
            elif refresh_status == 'Failed':
                refresh_completed = True
                print()  # New line after the \r
                print()
                
                # Get error details if available
                service_exception = latest_refresh.get('serviceExceptionJson', '')
                error_code = latest_refresh.get('errorCode', 'Unknown')
                
                error_msg = f"❌ REFRESH FAILED\n"
                error_msg += f"Model: {semantic_model_name}\n"
                error_msg += f"Error Code: {error_code}\n"
                if service_exception:
                    error_msg += f"Details: {service_exception}\n"
                
                print("─" * 80)
                print(error_msg)
                print("─" * 80)
                raise Exception(error_msg)
                
            elif refresh_status == 'Cancelled':
                refresh_completed = True
                print()  # New line after the \r
                print()
                error_msg = f"🚫 REFRESH CANCELLED: {semantic_model_name}"
                print("─" * 80)
                print(error_msg)
                print("─" * 80)
                raise Exception(error_msg)
            
            # If still in progress, wait before next poll
            if not refresh_completed:
                time.sleep(polling_interval)
                
        except requests.exceptions.RequestException as req_err:
            print(f"⚠️  Poll #{poll_count}: Network error - {str(req_err)}")
            time.sleep(polling_interval)
            continue
            
except KeyboardInterrupt:
    print()
    print("\n⚠️  Polling interrupted by user")
    raise
    
except Exception as e:
    if "TIMEOUT" in str(e) or "FAILED" in str(e) or "CANCELLED" in str(e):
        raise  # Re-raise our formatted errors
    else:
        error_msg = f"❌ ERROR during polling: {str(e)}"
        print(error_msg)
        raise Exception(error_msg)

In [ ]:
# Cell 6: Final Summary

print("=" * 80)
print("📊 EXECUTION SUMMARY")
print("=" * 80)

end_time = datetime.now()
total_elapsed = end_time - refresh_start_time
total_minutes = int(total_elapsed.total_seconds() / 60)
total_seconds = int(total_elapsed.total_seconds() % 60)

print(f"📊 Semantic Model:  {semantic_model_name}")
print(f"✅ Status:          SUCCESS")
print(f"⏰ Started:         {refresh_start_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"🏁 Completed:       {end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"⏱️  Total Duration:  {total_minutes}m {total_seconds}s")
print(f"🔍 Status Checks:   {poll_count}")
print()
print("=" * 80)
print("🎉 NOTEBOOK EXECUTION COMPLETE")
print("=" * 80)